In [99]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [100]:
data = pd.read_csv('spaceshi_titanic.csv')
data

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


In [101]:
def preprocess_spaceship_data(df):
    df = df.copy()

    df['Group'] = df['PassengerId'].apply(lambda x: x.split('_')[0])

    df['Cabin'] = df['Cabin'].fillna('Unknown/Unknown/Unknown')
    
    df[['Deck', 'Num', 'Side']] = df['Cabin'].str.split('/', expand=True)
    df['HasCabin'] = (df['Deck'] != 'Unknown').astype(int)
    df['Num'] = pd.to_numeric(df['Num'], errors='coerce').fillna(-1)

    expenses = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    
    for col in expenses:
        df.loc[(df['CryoSleep'] == True) & (df[col].isna()), col] = 0
    
    df['Total_Spend'] = df[expenses].fillna(0).sum(axis=1)
    
    df.loc[(df['Total_Spend'] > 0) & (df['CryoSleep'].isna()), 'CryoSleep'] = False

    df['NoSpending'] = (df['Total_Spend'] == 0).astype(int)
    
    for col in expenses:
        df[col] = df[col].fillna(0)

    for col in ['HomePlanet', 'Destination']:
        group_map = df.groupby('Group')[col].first()
        df[col] = df[col].fillna(df['Group'].map(group_map))
        df[col] = df[col].fillna(df[col].mode()[0])

    df['GroupSize'] = df.groupby('Group')['Group'].transform('count')
    
    df['IsAlone'] = (df['GroupSize'] == 1).astype(int)
    
    df['VIP'] = df['VIP'].fillna(False).astype(int)
    df['CryoSleep'] = df['CryoSleep'].fillna(False).astype(int)

    df['Age'] = df['Age'].fillna(df['Age'].median())

    df['Surname'] = df['Name'].str.split().str[-1].fillna('Unknown')

    df['FamilySize'] = df.groupby('Surname')['Surname'].transform('count')

    df = pd.get_dummies(df, columns=['HomePlanet', 'Destination', 'Deck', 'Side'], dtype='int8')

    cols_to_drop = ['PassengerId', 'Cabin', 'Name', 'Group', 'Surname']
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

    return df

In [102]:
X = data.drop("Transported", axis=1)
y = data["Transported"].astype(int)

In [103]:
x_train_raw, x_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [104]:
x_train = preprocess_spaceship_data(x_train_raw)
x_test = preprocess_spaceship_data(x_test_raw)

In [105]:
x_train, x_test = x_train.align(x_test, join='left', axis=1, fill_value=0)

In [113]:
# model = DecisionTreeClassifier(max_depth=4, min_samples_split=2, min_samples_leaf=1, random_state=42)
model = (n_neighbors=28, p=2)
# model = LogisticRegression(max_iter=10000, C=5)
model.fit(x_train, y_train)
acc_test = accuracy_score(y_test, model.predict(x_test))
acc_train = accuracy_score(y_train, model.predict(x_train))

print(f"Train Accuracy: {acc_train:.4f}")
print(f"Test Accuracy: {acc_test:.4f}")

Train Accuracy: 0.7977
Test Accuracy: 0.7821


In [114]:
test_df = pd.read_csv('spaceshi_titanic_test.csv')

# Сохраняем PassengerId отдельно (он нужен для финального файла)
passenger_ids = test_df['PassengerId']

# 2. Обрабатываем тестовые данные той же функцией
x_test_final = preprocess_spaceship_data(test_df)

# 3. Выравниваем колонки с обучающей выборкой (x_train)
# Это добавит недостающие колонки и удалит лишние, если они есть
x_train, x_test_final = x_train.align(x_test_final, join='left', axis=1, fill_value=0)

# 4. Делаем предсказание
predictions = model.predict(x_test_final)

In [115]:
# 5. Собираем результат в таблицу
submission = pd.DataFrame({
    "PassengerId": passenger_ids,
    "Transported": predictions.astype(bool) # Переводим 0/1 обратно в True/False
})

# 6. Сохраняем в CSV без индексов
submission.to_csv('submission.csv', index=False)

print("Файл submission.csv готов к отправке!")

Файл submission.csv готов к отправке!
